In [ ]:
import torch
from ttpi import TTPI
import numpy as np
np.set_printoptions(precision=2)
torch.set_printoptions(precision=2)
torch.set_default_dtype(torch.float64)

from dynamic_systems import PointMass
%load_ext autoreload
%autoreload 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
order = 2 # 1: 'velocity control', 2: 'acceleration control'

In [ ]:
dim = 2 # 2D point-mass
L = 1.0
position_max = torch.tensor([L]*dim).to(device)
position_min= -1*position_max

n_state = 100
n_action = 100

dt = 0.01 # time step

velocity_max = position_max/4
velocity_min = -1*velocity_max

acceleration_max = 1*velocity_max
acceleration_min = -1*acceleration_max

if order == 1:
    state_min = position_min 
    state_max = position_max
    action_min = velocity_min
    action_max = velocity_max

else:
    state_min = torch.concat((position_min,velocity_min),dim=-1)
    state_max = torch.concat((position_max,velocity_max),dim=-1)
    action_min = acceleration_min
    action_max = acceleration_max

domain_state = []
for i in range(len(state_max)):#x,dx
    domain_state.append(torch.linspace(state_min[i],state_max[i],n_state).to(device))


domain_action = []
for i in range(len(action_max)):#x,dx
    domain_action.append(torch.linspace(action_min[i],action_max[i],n_state).to(device))

# Add spherical obstacles in the space if any
x_obst = [torch.tensor([0.,-0.4]).to(device)]#[torch.tensor([0.,-0.5]).to(device)] #[torch.tensor([0.35,0.5]).to(device),torch.tensor([0.35,-0.3]).to(device),torch.tensor([-0.1,-0.55]).to(device)]
r_obst = [0.2] #[0.2]*len(x_obst)

sys = PointMass(order=order, dt=dt, dim=dim,
                x_obst=x_obst,r_obst=r_obst,
                w_obst=1e3, w_action=1, w_goal=1e3, w_scale=1, device=device)

In [ ]:
def forward_model(state,action):
    return sys.forward_simulate(state,action)

def reward(state,action):
    rewards, _ = sys.reward_state_action(state,action)
    return rewards

In [ ]:
# ttdp = None # defined in the later cell
def callback(ttpi,file_name='fig', callback_count=0):
    print("Testing....")
    state = torch.tensor([[0.,0.],
                          [-0.3,-0.],[-0.3,0.3],[-0.3,-0.3],
                          [0.3,-0.],[0.3,0.3],[0.3,-0.3],
                          [0,0.3],
                          [-0.75,-0.75],[-0.75,0.75],[0.75,-0.75],
                          [-0.6,-0.],[-0.6,0.2],[-0.5,-0.6],
                          [0.6,-0.2],[0.6,0.5],[0.6,-0.7],
                          [0.1,0.6],
                          [-0.8,-0.],[-0.8,0.7],[-0.8,-0.5],
                          [0.8,-0.],[0.,-0.9],[0.8,0.4],[0.8,-0.6],[0.9,0],
                          [0.1,0.8], [-0.1,-0.8], [0.1,-0.8], [0.2,-0.8]]).to(device)
    state = state
    if dim>2:
        state_append = torch.tensor([0.]*(dim-2)).to(device).view(1,-1).expand(state.shape[0],-1)
        state = torch.cat((state,state_append),dim=-1)
    if order==2:
        state = torch.cat((state,state*0),dim=-1) 
    history = []
    traj = state[:,:2].clone()[:,None,:]
    cum_reward = torch.tensor([0.]*state.shape[0]).to(device)
    for i in range(1000):
        action = ttpi.policy(state)
        r = ttpi.reward_normalized(state,action)
        cum_reward+=r
        state = forward_model(state,action)
        position = state[:,:dim]
        traj = torch.concat((traj,position[:,None,:2]),dim=1)
    print("Cumulative reward: ", cum_reward.mean())
    print("Avg: ", torch.mean(cum_reward))
    from plot_utils import plot_point_mass
    plt=plot_point_mass(traj.to('cpu'),
        xmax=L,
        x_target=torch.tensor([0,0]).to('cpu'), 
        x_obst=[x.to('cpu') for x in x_obst], r_obst=r_obst, 
        save_as = None,
        figsize=5)    
    plt.show()
    
    return r.mean().to("cpu"),cum_reward.mean().to("cpu")


In [ ]:
# callback(ttdp,file_name='fig')

In [ ]:
ttpi = TTPI(domain_state=domain_state, domain_action=domain_action, reward=reward, 
                forward_model=forward_model, 
                gamma=0.99,
                rmax_v=100, rmax_a=100, 
                nswp_v=10, nswp_a=10, 
                kickrank_v=5, kickrank_a=10,
                max_batch_v=10**4, max_batch_a=10**5,
                eps_cross_v=1e-3,eps_cross_a=1e-3,
                eps_round_v=1e-3,eps_round_a=1e-3, 
                n_samples=50,normalize_reward=False,
                verbose=True,
                device=device) 

In [ ]:

resume= False # resume=True => resume a previous training 
ttpi.train(n_iter_max=2000,resume=resume, 
        callback=callback, callback_freq=25,
        verbose=False, file_name='point_mass')

In [ ]:
.